# Lab 4

### Install required libraries

In [10]:
import os

# Make sure to install swig & mujoco as standalone system packages

os.environ['MUJOCO_PATH'] = '/usr'
os.environ['CMAKE_PREFIX_PATH'] = '/usr'
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

%pip install numpy==2.2.6 box2d-py mujoco gymnasium tqdm torch torchvision 'stable-baselines3[extra]' moviepy ipywidgets tensorboard --quiet

Note: you may need to restart the kernel to use updated packages.


### Setup
Load the libraries, environment and define constants.

In [16]:
import gymnasium as gym
import moviepy
import numpy as np
import torch
from pathlib import Path
from imageio import mimsave
import subprocess
from stable_baselines3 import SAC, PPO, DQN, __version__ as sb3_version
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.logger import configure


if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(
    f"Gymnasium: {gym.__version__}\n"
    f"SB3: {sb3_version}\n"
    f"Torch: {torch.__version__}\n"
    f"Device: {device}\n"
    f"MoviePy: {moviepy.__version__}"
)

# Environment configuration
REACHER_ENV_NAME = "Reacher-v5"
LUNAR_LANDER_ENV_NAME = "LunarLander-v3"

# Training hyperparameters
TOTAL_TRAINING_STEPS = 1_000_000
LEARNING_RATE = 3e-4
GAMMA = 0.99
VERBOSE = 0

# SAC-specific hyperparameters
SAC_BUFFER_SIZE = 1_000_000
SAC_BATCH_SIZE = 256
SAC_TAU = 0.005

# PPO-specific hyperparameters
NUM_PPO_STEPS = 2048
PPO_BATCH_SIZE = 64
GAE_LAMBDA = 0.95
NUM_PPO_EPOCHS = 10
PPO_CLIP_RANGE = 0.2

# Video recording configuration
RECORD_VIDEO = True
MAX_RENDER_STEPS = 2000  # Prevent infinite loops
VIDEO_FOLDER = Path("videos")
VIDEO_FOLDER.mkdir(parents=True, exist_ok=True)

# Evaluation configuration
SHOW_LOGS = True
LOG_DIR = Path("runs")

def make_env(name=REACHER_ENV_NAME, render_mode=None):
    env = gym.make(name, render_mode=render_mode)
    env = Monitor(env)
    return env

make_env_reacher = lambda: make_env(name=REACHER_ENV_NAME)
make_env_reacher_render = lambda: make_env(name=REACHER_ENV_NAME, render_mode="rgb_array")
make_env_lunar_lander = lambda: make_env(name=LUNAR_LANDER_ENV_NAME)
make_env_lunar_lander_render = lambda: make_env(name=LUNAR_LANDER_ENV_NAME, render_mode="rgb_array")

Gymnasium: 1.2.2
SB3: 2.7.0
Torch: 2.9.0+cu128
Device: cuda
MoviePy: 2.1.2


### Start log server

In [14]:
log_server_process = None
if SHOW_LOGS:
    # spawn a process to view logs in TensorBoard and save its PID to kill it later
    log_server_process = subprocess.Popen(
        [
            "tensorboard",
            "--logdir",
            str(LOG_DIR),
            "--host",
            "0.0.0.0",
            "--port",
            "6006",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        preexec_fn=os.setsid,
    )
    print("Log server started at http://localhost:6006")

Log server started at http://localhost:6006


### Soft Actor-Critic (SAC) Training on Reacher Environment

In [ ]:
vec_env_sac = DummyVecEnv([make_env_reacher])
vec_env_sac = VecNormalize(vec_env_sac, norm_obs=True, norm_reward=False)

if not Path("sac_reacher_model.zip").exists():
    logdir_sac = LOG_DIR / "sac_reacher"
    logdir_sac.mkdir(parents=True, exist_ok=True)

    model_sac = SAC(
        "MlpPolicy",
        vec_env_sac,
        verbose=VERBOSE,
        tensorboard_log=str(logdir_sac),
        learning_rate=LEARNING_RATE,
        buffer_size=SAC_BUFFER_SIZE,
        batch_size=SAC_BATCH_SIZE,
        tau=SAC_TAU,
        gamma=GAMMA,
        target_entropy="auto",
        device=device,
    )

    new_logger = configure(str(logdir_sac), ["tensorboard"])
    model_sac.set_logger(new_logger)

    model_sac.learn(total_timesteps=TOTAL_TRAINING_STEPS, progress_bar=True)
    model_sac.save("sac_reacher_model")
    
    vec_env_sac.save("vecnormalize_reacher.pkl")
    vec_env_sac.close()

### Proximal Policy Optimization (PPO) Training on Reacher Environment

In [ ]:
vec_env_ppo = DummyVecEnv([make_env_reacher])
vec_env_ppo = VecNormalize(vec_env_ppo, norm_obs=True, norm_reward=False)

if not Path("ppo_reacher_model.zip").exists():
    logdir_ppo = LOG_DIR / "ppo_reacher"
    logdir_ppo.mkdir(parents=True, exist_ok=True)

    model_ppo = PPO(
        "MlpPolicy",
        vec_env_ppo,
        verbose=VERBOSE,
        tensorboard_log=str(logdir_ppo),
        learning_rate=LEARNING_RATE,
        gamma=GAMMA,
        batch_size=PPO_BATCH_SIZE,
        n_steps=NUM_PPO_STEPS,
        gae_lambda=GAE_LAMBDA,
        n_epochs=NUM_PPO_EPOCHS,
        clip_range=PPO_CLIP_RANGE,
        device=device,
    )

    new_logger = configure(str(logdir_ppo), ["tensorboard"])
    model_ppo.set_logger(new_logger)

    model_ppo.learn(total_timesteps=TOTAL_TRAINING_STEPS, progress_bar=True)
    model_ppo.save("ppo_reacher_model")

    vec_env_ppo.save("vecnormalize_reacher.pkl")
    vec_env_ppo.close()

/mnt/shared/dev/labs/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


### Deep Q-Network (DQN) Training on LunarLander Environment

In [13]:
vec_env_dqn = DummyVecEnv([make_env_lunar_lander])
vec_env_dqn = VecNormalize(vec_env_dqn, norm_obs=True, norm_reward=True)

if not Path("dqn_lunar_lander_model.zip").exists():
    logdir_dqn = LOG_DIR / "dqn_lunar_lander"
    logdir_dqn.mkdir(parents=True, exist_ok=True)

    model_dqn = DQN(
        "MlpPolicy",
        vec_env_dqn,
        verbose=VERBOSE,
        tensorboard_log=str(logdir_dqn),
        learning_rate=LEARNING_RATE,
        buffer_size=1_000_000,
        batch_size=64,
        gamma=GAMMA,
        device=device,
    )

    new_logger = configure(str(logdir_dqn), ["tensorboard"])
    model_dqn.set_logger(new_logger)

    model_dqn.learn(total_timesteps=TOTAL_TRAINING_STEPS, progress_bar=True)
    model_dqn.save("dqn_lunar_lander_model")

    vec_env_dqn.save("vecnormalize_lunar_lander.pkl")
    vec_env_dqn.close()

/mnt/shared/dev/labs/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Output()

### Stop log server

In [19]:
if log_server_process is not None:
    log_server_process.terminate()
    print("Log server stopped.")

Log server stopped.


### Evaluate Model and record a Video

In [ ]:
def evaluate_model(model_path, vec_path, n_episodes=10, record_video=RECORD_VIDEO):
    algo_name = (
        "sac"
        if "sac" in model_path.lower()
        else "ppo" if "ppo" in model_path.lower() else "dqn"
    )

    # Create and wrap environment
    make_env_func = (
        make_env_reacher_render
        if algo_name in ["sac", "ppo"]
        else make_env_lunar_lander_render
    )
    env = DummyVecEnv([make_env_func])
    env = VecNormalize.load(vec_path, env)
    env.training = False
    env.norm_reward = False

    # Load model
    model = (
        SAC.load(model_path, env=env)
        if algo_name == "sac"
        else PPO.load(model_path, env=env)
        if algo_name == "ppo"
        else DQN.load(model_path, env=env)
    )

    episode_rewards = []
    frames = []

    for episode in range(1, n_episodes + 1):
        obs = env.reset()
        total_reward = 0
        done = False
        steps = 0

        while not done and steps < MAX_RENDER_STEPS:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated = env.step(action)

            # Capture frame for first episode only
            if record_video:
                frame = env.envs[0].render()
                if frame is not None:
                    frames.append(frame)

            total_reward += float(np.sum(reward))

            done = np.any(terminated)
            steps += 1

        episode_rewards.append(total_reward)
        print(
            f"Episode {episode} finished after {steps} steps, reward: {total_reward:.2f}"
        )

    # Save video if frames were collected
    if record_video and len(frames) > 0:
        video_path = str(VIDEO_FOLDER) + f"/{algo_name}_eval.mp4"
        print(f"Video will be saved to: {video_path}")

        print(f"Saving video with {len(frames)} frames...")
        mimsave(video_path, frames, fps=30)
        print(f"✅ {algo_name.upper()} video saved to: {video_path}")
    elif record_video:
        print(f"⚠️ No frames captured for {algo_name.upper()}")

    env.close()
    return np.mean(episode_rewards)


# Run evaluation with video recording
print("🎬 Recording SAC video...")
sac_reward = evaluate_model(
    "sac_reacher_model.zip", "vecnormalize_reacher.pkl", record_video=True
)

print("\n🎬 Recording PPO video...")
ppo_reward = evaluate_model(
    "ppo_reacher_model.zip", "vecnormalize_reacher_ppo.pkl", record_video=True
)

print("\n🎬 Recording DQN video...")
dqn_reward = evaluate_model(
    "dqn_lunar_lander_model.zip", "vecnormalize_lunar_lander.pkl", record_video=True
)

print(f"\n{'='*50}")
print(f"SAC mean reward: {sac_reward:.2f}")
print(f"PPO mean reward: {ppo_reward:.2f}")
print(f"DQN mean reward: {dqn_reward:.2f}")
print(f"{'='*50}")

🎬 Recording SAC video...



(python:294085): Gtk-WARNING **: 14:18:11.214: gtk_disable_setlocale() must be called before gtk_init()


Episode 1 finished after 50 steps, reward: -2.16
Episode 2 finished after 50 steps, reward: -2.24
Episode 3 finished after 50 steps, reward: -5.52
Episode 4 finished after 50 steps, reward: -2.33
Episode 5 finished after 50 steps, reward: -3.68
Episode 6 finished after 50 steps, reward: -3.56
Episode 7 finished after 50 steps, reward: -5.21
Episode 8 finished after 50 steps, reward: -1.33
Episode 9 finished after 50 steps, reward: -3.88
Episode 10 finished after 50 steps, reward: -4.47
Video will be saved to: videos/sac_reacher_eval.mp4
Saving video with 500 frames...
✅ SAC video saved to: videos/sac_reacher_eval.mp4

🎬 Recording PPO video...


/mnt/shared/dev/labs/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(

(python:294085): Gtk-WARNING **: 14:18:16.306: gtk_disable_setlocale() must be called before gtk_init()


Episode 1 finished after 50 steps, reward: -3.24
Episode 2 finished after 50 steps, reward: -5.48
Episode 3 finished after 50 steps, reward: -5.13
Episode 4 finished after 50 steps, reward: -3.78
Episode 5 finished after 50 steps, reward: -4.61
Episode 6 finished after 50 steps, reward: -2.71
Episode 7 finished after 50 steps, reward: -4.11
Episode 8 finished after 50 steps, reward: -2.75
Episode 9 finished after 50 steps, reward: -2.42
Episode 10 finished after 50 steps, reward: -4.77
Video will be saved to: videos/ppo_reacher_eval.mp4
Saving video with 500 frames...
✅ PPO video saved to: videos/ppo_reacher_eval.mp4

🎬 Recording DQN video...
Episode 1 finished after 639 steps, reward: 241.46
Episode 2 finished after 660 steps, reward: 235.54
Episode 3 finished after 467 steps, reward: 225.52
Episode 4 finished after 1000 steps, reward: 122.47
Episode 5 finished after 334 steps, reward: 200.71
Episode 6 finished after 342 steps, reward: 243.42
Episode 7 finished after 438 steps, reward

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Episode 10 finished after 841 steps, reward: 227.62
Video will be saved to: videos/dqn_reacher_eval.mp4
Saving video with 6084 frames...
✅ DQN video saved to: videos/dqn_reacher_eval.mp4

SAC mean reward: -3.44
PPO mean reward: -3.90
DQN mean reward: 215.59
